In [54]:
import MetaTrader5 as mt5

In [55]:
mt5.initialize()

True

In [56]:
symbol = "EURUSD"
timeframe = mt5.TIMEFRAME_M30
num_candles = 100

In [57]:
rates = mt5.copy_rates_from_pos(symbol, timeframe,0, num_candles)

In [58]:
import numpy as np
import pandas as pd

df = pd.DataFrame(rates)

In [59]:
df['time'] = pd.to_datetime(df['time'], unit='s')

In [60]:
df

,time,open,high,low,close,tick_volume,spread,real_volume
0,2024-11-22 16:30:00,1.04115,1.04257,1.04024,1.04048,3320,0,0
1,2024-11-22 17:00:00,1.04045,1.04202,1.03951,1.04127,3705,0,0
2,2024-11-22 17:30:00,1.04127,1.04183,1.04016,1.04107,3168,0,0
3,2024-11-22 18:00:00,1.04107,1.04119,1.03920,1.03956,2396,0,0
4,2024-11-22 18:30:00,1.03956,1.03999,1.03936,1.03984,1312,0,0
...,...,...,...,...,...,...,...,...
95,2024-11-26 16:00:00,1.05037,1.05061,1.04897,1.04994,2341,0,0
96,2024-11-26 16:30:00,1.04994,1.05053,1.04923,1.05028,2652,0,0
97,2024-11-26 17:00:00,1.05028,1.05100,1.04868,1.04870,2884,0,0
98,2024-11-26 17:30:00,1.04870,1.05016,1.04786,1.04796,2867,0,0


In [61]:
df.to_csv("check.csv")

In [62]:
def calculate_macd(data, short_period=12, long_period=26, signal_period=9):
    """Calculate the MACD and signal line."""
    data['ema_short'] = data['close'].ewm(span=short_period, adjust=False).mean()
    data['ema_long'] = data['close'].ewm(span=long_period, adjust=False).mean()
    data['MACD'] = data['ema_short'] - data['ema_long']
    data['signal_line'] = data['MACD'].ewm(span=signal_period, adjust=False).mean()
    data['MACD_histogram'] = data['MACD'] - data['signal_line']
    return data


def generate_macd_trend(df):
    df['Trend'] = 0

    # Generate buy (1) and sell (-1) Trend
    for i in range(1, len(df)):
        # Bullish crossover (Buy)
        if df.loc[i, 'MACD'] > df.loc[i, 'signal_line']:
            df.loc[i, 'Trend'] = 1

        # Bearish crossover (Sell)
        elif df.loc[i, 'MACD'] <= df.loc[i, 'signal_line']:
            df.loc[i, 'Trend'] = 0

    return df

def check_trend(data, short_period=12, long_period=26, signal_period=9):
    """
    Check if a MACD crossover occurred on the previous candle for the given symbol and timeframe.
    
    Parameters:
        symbol (str): The trading symbol (e.g., "EURUSD").
        timeframe: MetaTrader 5 timeframe (e.g., mt5.TIMEFRAME_M1).
        short_period (int): Short period for the MACD calculation. Default is 12.
        long_period (int): Long period for the MACD calculation. Default is 26.
        signal_period (int): Signal line period for the MACD calculation. Default is 9.
    
    Returns:
        bool: True if a MACD crossover occurred, False otherwise.
    """

    
    # Convert rates to DataFrame
    # data.set_index('time', inplace=True)
    
    # Calculate MACD and signal line
    data = calculate_macd(data, short_period, long_period, signal_period)
    
    # Generate signals
    data = generate_macd_trend(data)
    
    # Check for crossover in the last complete candle (second-to-last row)
    if len(data) < 2:
        return False  # Not enough data to determine
    
    return data  # True if there was a signal in the previous candle


In [63]:
check_trend(df)

,time,open,high,low,close,tick_volume,spread,real_volume,ema_short,ema_long,MACD,signal_line,MACD_histogram,Trend
0,2024-11-22 16:30:00,1.04115,1.04257,1.04024,1.04048,3320,0,0,1.040480,1.040480,0.000000e+00,0.000000,0.000000,0
1,2024-11-22 17:00:00,1.04045,1.04202,1.03951,1.04127,3705,0,0,1.040602,1.040539,6.301994e-05,0.000013,0.000050,1
2,2024-11-22 17:30:00,1.04127,1.04183,1.04016,1.04107,3168,0,0,1.040674,1.040578,9.572195e-05,0.000029,0.000066,1
3,2024-11-22 18:00:00,1.04107,1.04119,1.03920,1.03956,2396,0,0,1.040502,1.040502,-2.035078e-07,0.000023,-0.000024,0
4,2024-11-22 18:30:00,1.03956,1.03999,1.03936,1.03984,1312,0,0,1.040400,1.040453,-5.302028e-05,0.000008,-0.000061,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2024-11-26 16:00:00,1.05037,1.05061,1.04897,1.04994,2341,0,0,1.050468,1.049639,8.294707e-04,0.000808,0.000022,1
96,2024-11-26 16:30:00,1.04994,1.05053,1.04923,1.05028,2652,0,0,1.050439,1.049686,7.530193e-04,0.000797,-0.000044,0
97,2024-11-26 17:00:00,1.05028,1.05100,1.04868,1.04870,2884,0,0,1.050172,1.049613,5.585002e-04,0.000749,-0.000191,0
98,2024-11-26 17:30:00,1.04870,1.05016,1.04786,1.04796,2867,0,0,1.049831,1.049491,3.407031e-04,0.000667,-0.000327,0


In [64]:
def calculate_macd(data, short_period=12, long_period=26, signal_period=9):
    """Calculate the MACD, signal line, and histogram."""
    data['ema_short'] = data['close'].ewm(span=short_period, adjust=False).mean()
    data['ema_long'] = data['close'].ewm(span=long_period, adjust=False).mean()
    data['MACD'] = data['ema_short'] - data['ema_long']
    data['signal_line'] = data['MACD'].ewm(span=signal_period, adjust=False).mean()
    data['MACD_histogram'] = data['MACD'] - data['signal_line']
    return data

def generate_macd_trend(df):
    """Generate MACD trend (Buy = 1, Sell = -1, No Trend = 0)."""
    df['Trend'] = 0  # Default trend value
    df.loc[df['MACD'] > df['signal_line'], 'Trend'] = 1  # Bullish crossover
    df.loc[df['MACD'] < df['signal_line'], 'Trend'] = -1  # Bearish crossover
    return df


def check_trend(data, short_period=12, long_period=26, signal_period=9):
    """
    Check if a MACD crossover occurred on the previous candle for the given symbol and timeframe.
    
    Parameters:
        symbol (str): The trading symbol (e.g., "EURUSD").
        timeframe: MetaTrader 5 timeframe (e.g., mt5.TIMEFRAME_M1).
        short_period (int): Short period for the MACD calculation. Default is 12.
        long_period (int): Long period for the MACD calculation. Default is 26.
        signal_period (int): Signal line period for the MACD calculation. Default is 9.
    
    Returns:
        bool: True if a MACD crossover occurred, False otherwise.
    """

    
    # Convert rates to DataFrame
    # data.set_index('time', inplace=True)
    
    # Calculate MACD and signal line
    data = calculate_macd(data, short_period, long_period, signal_period)
    
    # Generate signals
    data = generate_macd_trend(data)
    
    # Check for crossover in the last complete candle (second-to-last row)
    if len(data) < 2:
        return False  # Not enough data to determine
    
    return data  # True if there was a signal in the previous candle


In [65]:
check_trend(df)


,time,open,high,low,close,tick_volume,spread,real_volume,ema_short,ema_long,MACD,signal_line,MACD_histogram,Trend
0,2024-11-22 16:30:00,1.04115,1.04257,1.04024,1.04048,3320,0,0,1.040480,1.040480,0.000000e+00,0.000000,0.000000,0
1,2024-11-22 17:00:00,1.04045,1.04202,1.03951,1.04127,3705,0,0,1.040602,1.040539,6.301994e-05,0.000013,0.000050,1
2,2024-11-22 17:30:00,1.04127,1.04183,1.04016,1.04107,3168,0,0,1.040674,1.040578,9.572195e-05,0.000029,0.000066,1
3,2024-11-22 18:00:00,1.04107,1.04119,1.03920,1.03956,2396,0,0,1.040502,1.040502,-2.035078e-07,0.000023,-0.000024,-1
4,2024-11-22 18:30:00,1.03956,1.03999,1.03936,1.03984,1312,0,0,1.040400,1.040453,-5.302028e-05,0.000008,-0.000061,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2024-11-26 16:00:00,1.05037,1.05061,1.04897,1.04994,2341,0,0,1.050468,1.049639,8.294707e-04,0.000808,0.000022,1
96,2024-11-26 16:30:00,1.04994,1.05053,1.04923,1.05028,2652,0,0,1.050439,1.049686,7.530193e-04,0.000797,-0.000044,-1
97,2024-11-26 17:00:00,1.05028,1.05100,1.04868,1.04870,2884,0,0,1.050172,1.049613,5.585002e-04,0.000749,-0.000191,-1
98,2024-11-26 17:30:00,1.04870,1.05016,1.04786,1.04796,2867,0,0,1.049831,1.049491,3.407031e-04,0.000667,-0.000327,-1


In [66]:
import pandas as pd
import pandas_ta as ta

def calculate_macd(data, short_period=12, long_period=26, signal_period=9):
    """
    Calculate the MACD, signal line, and histogram using pandas_ta.
    """
    macd = ta.macd(data['close'], fast=short_period, slow=long_period, signal=signal_period)
    data['MACD'] = macd['MACD_12_26_9']
    data['signal_line'] = macd['MACDs_12_26_9']
    data['MACD_histogram'] = macd['MACDh_12_26_9']
    return data

def generate_macd_trend(data):
    """
    Generate MACD trend (Buy = 1, Sell = -1, No Trend = 0).
    """
    data['Trend'] = 0  # Default trend value
    data.loc[data['MACD'] > data['signal_line'], 'Trend'] = 1  # Bullish crossover
    data.loc[data['MACD'] < data['signal_line'], 'Trend'] = -1  # Bearish crossover
    return data

def check_trend(data, short_period=12, long_period=26, signal_period=9):
    """
    Check if a MACD crossover occurred on the previous candle for the given symbol and timeframe.
    
    Parameters:
        symbol (str): The trading symbol (e.g., "EURUSD").
        timeframe: MetaTrader 5 timeframe (e.g., mt5.TIMEFRAME_M1).
        short_period (int): Short period for the MACD calculation. Default is 12.
        long_period (int): Long period for the MACD calculation. Default is 26.
        signal_period (int): Signal line period for the MACD calculation. Default is 9.
    
    Returns:
        bool: True if a MACD crossover occurred, False otherwise.
    """

    
    # Convert rates to DataFrame
    # data.set_index('time', inplace=True)
    
    # Calculate MACD and signal line
    data = calculate_macd(data, short_period, long_period, signal_period)
    
    # Generate signals
    data = generate_macd_trend(data)
    
    # Check for crossover in the last complete candle (second-to-last row)
    if len(data) < 2:
        return False  # Not enough data to determine
    
    return data  # True if there was a signal in the previous candle


In [70]:
check_trend(df)


,time,open,high,low,close,tick_volume,spread,real_volume,ema_short,ema_long,MACD,signal_line,MACD_histogram,Trend
0,2024-11-22 16:30:00,1.04115,1.04257,1.04024,1.04048,3320,0,0,1.041068,1.044171,-0.003103,-0.003103,0.000000,0
1,2024-11-22 17:00:00,1.04045,1.04202,1.03951,1.04127,3705,0,0,1.041068,1.044171,-0.003103,-0.003103,0.000000,0
2,2024-11-22 17:30:00,1.04127,1.04183,1.04016,1.04107,3168,0,0,1.041068,1.044171,-0.003103,-0.003103,0.000000,0
3,2024-11-22 18:00:00,1.04107,1.04119,1.03920,1.03956,2396,0,0,1.041068,1.044171,-0.003103,-0.003103,0.000000,0
4,2024-11-22 18:30:00,1.03956,1.03999,1.03936,1.03984,1312,0,0,1.041068,1.044171,-0.003103,-0.003103,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2024-11-26 16:00:00,1.05037,1.05061,1.04897,1.04994,2341,0,0,1.050468,1.049634,0.000834,0.000814,0.000020,1
96,2024-11-26 16:30:00,1.04994,1.05053,1.04923,1.05028,2652,0,0,1.050439,1.049682,0.000757,0.000803,-0.000046,-1
97,2024-11-26 17:00:00,1.05028,1.05100,1.04868,1.04870,2884,0,0,1.050172,1.049609,0.000562,0.000755,-0.000192,-1
98,2024-11-26 17:30:00,1.04870,1.05016,1.04786,1.04796,2867,0,0,1.049831,1.049487,0.000344,0.000673,-0.000328,-1


In [69]:
import pandas as pd
import pandas_ta as ta


def calculate_ema_with_sma(data, period):
    """
    Custom function for EMA where the first value is SMA, similar to MetaTrader.
    """
    # First period as SMA
    sma = data[:period].mean()
    ema = data.copy()
    ema[:period] = sma  # Replace the first 'period' values with SMA
    
    # Apply the standard EMA formula for the rest of the data
    for i in range(period, len(data)):
        ema[i] = (data[i] * (2 / (period + 1))) + (ema[i-1] * (1 - (2 / (period + 1))))
    return ema

def calculate_macd(data, short_period=12, long_period=26, signal_period=9):
    """
    Calculate MACD with the custom SMA-to-EMA initialization approach.
    """
    data['ema_short'] = calculate_ema_with_sma(data['close'], short_period)
    data['ema_long'] = calculate_ema_with_sma(data['close'], long_period)
    data['MACD'] = data['ema_short'] - data['ema_long']
    data['signal_line'] = calculate_ema_with_sma(data['MACD'], signal_period)
    data['MACD_histogram'] = data['MACD'] - data['signal_line']
    return data


def generate_macd_trend(data):
    """
    Generate MACD trend (Buy = 1, Sell = -1, No Trend = 0).
    """
    data['Trend'] = 0  # Default trend value
    data.loc[data['MACD'] > data['signal_line'], 'Trend'] = 1  # Bullish crossover
    data.loc[data['MACD'] < data['signal_line'], 'Trend'] = -1  # Bearish crossover
    return data

def check_trend(data, short_period=12, long_period=26, signal_period=9):
    """
    Check if a MACD crossover occurred on the previous candle for the given symbol and timeframe.
    
    Parameters:
        symbol (str): The trading symbol (e.g., "EURUSD").
        timeframe: MetaTrader 5 timeframe (e.g., mt5.TIMEFRAME_M1).
        short_period (int): Short period for the MACD calculation. Default is 12.
        long_period (int): Long period for the MACD calculation. Default is 26.
        signal_period (int): Signal line period for the MACD calculation. Default is 9.
    
    Returns:
        bool: True if a MACD crossover occurred, False otherwise.
    """

    
    # Convert rates to DataFrame
    # data.set_index('time', inplace=True)
    
    # Calculate MACD and signal line
    data = calculate_macd(data, short_period, long_period, signal_period)
    
    # Generate signals
    data = generate_macd_trend(data)
    
    # Check for crossover in the last complete candle (second-to-last row)
    if len(data) < 2:
        return False  # Not enough data to determine
    
    return data  # True if there was a signal in the previous candle
